In [48]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: C:\\Users\\jaskew\\Documents\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [49]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_HOURS = 48  # rolling wall-clock window in UTC (not calendar midnights)
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Single cutoff instant for TQL, observed_src filter, and workbook filter
_now_utc = datetime.now(pytz.UTC)
_last_observed_cutoff_dt = _now_utc - timedelta(hours=QUERY_LOOKBACK_HOURS)
start = _last_observed_cutoff_dt.strftime("%Y-%m-%dT%H:%M:%SZ")
LAST_OBSERVED_CUTOFF_TS = pd.Timestamp(_last_observed_cutoff_dt)

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src["lastObserved"] >= LAST_OBSERVED_CUTOFF_TS]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
    # Keep rows where top-level rating >= 3 OR coalesced threatAssessRating >= 3, and confidence >= 50.
    # Coalesce flat vs nested threatAssess columns per row (do not fold `rating` into threatAssessRating).
    _rating_cols = ("threatAssessRating", "threatAssess.threatAssessRating")
    _confidence_cols = ("threatAssessConfidence", "threatAssess.threatAssessConfidence")

    def _first_non_null_numeric(df, ordered_cols):
        present = [c for c in ordered_cols if c in df.columns]
        if not present:
            return None
        out = pd.to_numeric(df[present[0]], errors="coerce")
        for c in present[1:]:
            s = pd.to_numeric(df[c], errors="coerce")
            out = out.mask(out.isna(), s)
        return out

    _tar = _first_non_null_numeric(observed_src, _rating_cols)
    _tc = _first_non_null_numeric(observed_src, _confidence_cols)
    if _tar is None or _tc is None:
        raise KeyError(
            f"Could not resolve Threat Assess columns. Tried rating={_rating_cols}, "
            f"confidence={_confidence_cols}. Columns: {list(observed_src.columns)}"
        )
    if "rating" in observed_src.columns:
        _r = pd.to_numeric(observed_src["rating"], errors="coerce")
    else:
        _r = pd.Series(float("nan"), index=observed_src.index, dtype=float)
    _pre_ta = len(observed_src)
    # Use >= 50 so a boundary value of 50.0 is included (strict > 50 dropped those rows).
    _pass_rating_band = (_tar >= 3) | (_r >= 3)
    observed_src = observed_src[_pass_rating_band & (_tc >= 50)].copy()
    display(
        f"Threat assess filter ((rating>=3 OR threatAssessRating>=3), confidence>=50) coalescing {_rating_cols} / {_confidence_cols}: "
        f"{_pre_ta} -> {len(observed_src)} rows."
    )
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

"Threat assess filter ((rating>=3 OR threatAssessRating>=3), confidence>=50) coalescing ('threatAssessRating', 'threatAssess.threatAssessRating') / ('threatAssessConfidence', 'threatAssess.threatAssessConfidence'): 800 -> 232 rows."

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,indicator,sources
1,6755399501091083,2026-03-26T11:44:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-22T11:20:03Z,3.0,59.0,2.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...","[{'id': 5629499582001531, 'dateAdded': '2026-0...",NaN,NaN,NaN,NaN,NaN,196.11.86.134,"CMS_CTI, HTOC Org"
2,6755399503037237,2026-04-07T12:09:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-22T11:20:02Z,3.0,80.0,1.75,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,NaN,207.90.244.21,HTOC Org
3,6755399501091096,2026-03-26T11:44:41Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-22T11:20:02Z,3.0,79.0,1.50,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,NaN,87.236.176.163,HTOC Org
4,6755399487004202,2026-01-14T14:12:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-22T11:20:02Z,3.0,72.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,NaN,172.105.147.201,HTOC Org
6,6755399487003707,2026-01-13T13:34:19Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-22T11:20:02Z,5.0,93.0,2.50,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...","[{'id': 6755399477000414, 'dateAdded': '2026-0...",NaN,NaN,NaN,NaN,NaN,8.209.96.179,"CMS_CTI, HTOC Org"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1108,5629499537015458,2025-04-23T15:01:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-16T09:55:51Z,5.0,53.0,2.50,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...","[{'id': 6755399444000513, 'dateAdded': '2025-0...",NaN,NaN,NaN,NaN,NaN,169.150.203.29,"CMS_CTI, HTOC Org"
1113,6755399501091090,2026-03-26T11:44:36Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-16T09:19:24Z,3.0,79.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,NaN,185.247.137.154,HTOC Org
1129,6755399493005644,2026-02-26T14:09:02Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-15T23:19:03Z,5.0,65.0,5.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...","[{'id': 6755399483006350, 'dateAdded': '2026-0...",NaN,NaN,NaN,NaN,NaN,148.72.168.81,HTOC Org
1139,5629499580023419,2025-12-11T14:47:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-15T12:18:54Z,3.0,68.0,3.00,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2076981, 'name': 'SOAR Indicator PB', ...",NaN,NaN,NaN,NaN,NaN,NaN,128.231.73.154,HTOC Org


In [50]:
observed_src[observed_src['indicator'] == '196.11.86.134']

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,legacyLink,tags.data,associatedGroups.data,hostName,dnsActive,whoisActive,source,address,indicator,sources
1,6755399501091083,2026-03-26T11:44:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-22T11:20:03Z,3.0,59.0,2.0,...,https://hvs.threatconnect.com/auth/indicators/...,"[{'id': 2023876, 'name': 'VA OIT CSOC CTS Splu...","[{'id': 5629499582001531, 'dateAdded': '2026-0...",NaN,NaN,NaN,NaN,NaN,196.11.86.134,"CMS_CTI, HTOC Org"


In [51]:
import pandas as pd
import ast
from datetime import datetime, timedelta
import pytz

# Load the Excel file
file_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"
df = pd.read_excel(file_path)


# Keep only indicators that are also in observed_src
_indicator_col = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
if _indicator_col is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")

_observed_indicators = set(observed_src["indicator"].dropna().astype(str))
df = df[df[_indicator_col].astype(str).isin(_observed_indicators)].copy()

# Last Observed column: values come only from observed_src (ThreatConnect), not the workbook
_last_observed_col = next(
    (
        c
        for c in [
            "Last Observed",
            "lastObserved",
            "LastObserved",
            "last_observed",
            "LAST OBSERVED",
        ]
        if c in df.columns
    ),
    None,
)
if _last_observed_col is None:
    raise KeyError(f"Could not find 'Last Observed' column in df. Columns: {list(df.columns)}")

_assoc_groups_src_col = "associatedGroups.data"
_assoc_groups_target_col = "Associated Groups"
if _assoc_groups_src_col not in observed_src.columns:
    raise KeyError(
        f"Could not find '{_assoc_groups_src_col}' column in observed_src. Columns: {list(observed_src.columns)}"
    )


def _extract_group_ids(value):
    # Handle scalar nulls safely; avoid pd.isna on list-like values.
    if value is None:
        return pd.NA

    parsed = value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return pd.NA
        try:
            parsed = ast.literal_eval(text)
        except (ValueError, SyntaxError):
            return text
    elif isinstance(value, float) and pd.isna(value):
        return pd.NA

    if isinstance(parsed, dict):
        gid = parsed.get("id")
        return f"Group Id: {gid}" if gid is not None else pd.NA

    if isinstance(parsed, list):
        ids = []
        for item in parsed:
            if isinstance(item, dict) and item.get("id") is not None:
                ids.append(f"Group Id: {item.get('id')}")
        return ", ".join(ids) if ids else pd.NA

    return pd.NA


_observed_latest = (
    observed_src.dropna(subset=["indicator"])
    .assign(
        indicator=lambda d: d["indicator"].astype(str),
        lastObserved=lambda d: pd.to_datetime(d["lastObserved"], utc=True, errors="coerce"),
    )
    .sort_values("lastObserved")
    .drop_duplicates(subset=["indicator"], keep="last")
)

_last_obs_by_indicator = _observed_latest.set_index("indicator")["lastObserved"]
_assoc_groups_by_indicator = _observed_latest.set_index("indicator")[_assoc_groups_src_col].map(_extract_group_ids)

# Last Observed: only from ThreatConnect (observed_src); do not fall back to Excel dates
_df_ind = df[_indicator_col].astype(str)
df[_last_observed_col] = pd.to_datetime(_df_ind.map(_last_obs_by_indicator), utc=True, errors="coerce")
_qh = QUERY_LOOKBACK_HOURS if "QUERY_LOOKBACK_HOURS" in globals() else 48
_last_obs_cutoff = (
    LAST_OBSERVED_CUTOFF_TS
    if "LAST_OBSERVED_CUTOFF_TS" in globals()
    else pd.Timestamp(datetime.now(pytz.UTC) - timedelta(hours=_qh), tz="UTC")
)
_pre_lo = len(df)
df = df[df[_last_observed_col].notna() & (df[_last_observed_col] >= _last_obs_cutoff)].copy()
display(
    f"Last Observed filter (ThreatConnect only, >= {_last_obs_cutoff}): {_pre_lo} -> {len(df)} rows."
)

_df_ind = df[_indicator_col].astype(str)

# Add associatedGroups.data ids from observed_src by indicator, stored as 'Associated Groups'
if _assoc_groups_target_col in df.columns:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator).combine_first(df[_assoc_groups_target_col])
else:
    df[_assoc_groups_target_col] = _df_ind.map(_assoc_groups_by_indicator)

df

'Last Observed filter (ThreatConnect only, >= 2026-04-20 11:45:25.327668+00:00): 230 -> 230 rows.'

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups
2,1.192.18.4,2026-04-21 00:00:00+00:00,Address,18,3,0.999014,0,0,"CMS, FDA, HRSA, OS",NaN,NaN,190,373,420,medium,[2026-04-20] Severity: medium. VT score: 9. Co...,<NA>
28,103.120.116.162,2026-04-22 00:00:00+00:00,Address,83,3,0.995452,0,0,"CMS, DHA, FDA, HRSA, NIH, OS, VA",Incident:INC9385644,NaN,180,590,644,high,[2026-04-20] Severity: high. VT score: 11. Con...,<NA>
45,103.140.62.43,2026-04-22 00:00:00+00:00,Address,24,5,0.998685,0,0,"DHA, OS, VA",NaN,NaN,180,590,149,low,[2026-04-20] Severity: low. VT score: 3. Conte...,Group Id: 315446
208,104.28.243.105,2026-04-21 00:00:00+00:00,Address,74,5,0.995945,0,0,"CMS, DHA, FDA, HHS, HRSA, NIH, OS, VA",NaN,NaN,180,590,155,low,[2026-04-20] Severity: low. VT score: 0. Conte...,"Group Id: 6755399457034470, Group Id: 56294995..."
213,104.41.137.249,2026-04-22 00:00:00+00:00,Address,11,3,0.999397,0,0,"CMS, FDA, HRSA, IHS, OS, VA",Incident:INC9486336;Incident:INC9486332,NaN,480,740,217,medium,[2026-04-20] Severity: low. VT score: 1. Conte...,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3453,checkmarx.zone,2026-04-22 00:00:00+00:00,Host,22,5,0.998795,0,0,NIH,NaN,TeamPCP,890,824,153,low,[2026-04-20] Severity: low. VT score not avail...,"Group Id: 6755399491002373, Group Id: 56294995..."
3466,ctrk.klclick3.com,2026-04-22 00:00:00+00:00,Host,48,3,0.997370,0,0,"CMS, DHA, FDA, HRSA, IHS, NIH",NaN,NaN,180,590,78,low,[2026-04-20] Severity: low. VT score not avail...,Group Id: 6755399484001259
3543,link.storjshare.io,2026-04-22 00:00:00+00:00,Host,39,4,0.997863,0,0,"CMS, DHA, IHS, NIH",NaN,Banished Kitten,890,824,216,medium,[2026-04-20] Severity: low. VT score not avail...,"Group Id: 6755399488000346, Group Id: 56294995..."
3614,pulse.herosms.cc,2026-04-22 00:00:00+00:00,Host,94,4,0.994849,0,0,"DHA, NIH, VA",Incident:6755399479000129;Incident:67553994780...,NaN,190,452,204,medium,[2026-04-20] Severity: medium. VT score not av...,"Group Id: 6755399481021003, Group Id: 67553994..."


In [52]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=2)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260422.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260421.csv']
Loaded data from 2 files.


C:\Users\jaskew\AppData\Local\Temp\ipykernel_48144\1484081131.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


In [53]:
# Keep only indicators that are present in observed_data_df and seen by 2+ OpDiv partners
_indicator_col_df = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in df.columns), None)
_indicator_col_obs = next((c for c in ["indicator", "Indicator", "INDICATOR"] if c in observed_data_df.columns), None)
_opdiv_col = next((c for c in ["OpDiv", "opdiv", "OPDIV"] if c in observed_data_df.columns), None)

if _indicator_col_df is None:
    raise KeyError(f"Could not find indicator column in df. Columns: {list(df.columns)}")
if _indicator_col_obs is None:
    raise KeyError(f"Could not find indicator column in observed_data_df. Columns: {list(observed_data_df.columns)}")
if _opdiv_col is None:
    raise KeyError(f"Could not find OpDiv column in observed_data_df. Columns: {list(observed_data_df.columns)}")

obs = observed_data_df.dropna(subset=[_indicator_col_obs, _opdiv_col]).copy()
obs[_indicator_col_obs] = obs[_indicator_col_obs].astype(str).str.strip()
obs[_opdiv_col] = obs[_opdiv_col].astype(str).str.strip()

partners_by_indicator = (
    obs.groupby(_indicator_col_obs)[_opdiv_col]
    .apply(lambda s: sorted(set(x for x in s if x)))
)

eligible_partners = partners_by_indicator[partners_by_indicator.str.len() >= 2]
opdiv_map = eligible_partners.apply(lambda vals: ", ".join(vals))

last_24h_multiple_partners = df[
    df[_indicator_col_df].astype(str).str.strip().isin(eligible_partners.index)
].copy()
last_24h_multiple_partners["OpDiv"] = (
    last_24h_multiple_partners[_indicator_col_df].astype(str).str.strip().map(opdiv_map)
)
last_24h_multiple_partners["Partners"] = last_24h_multiple_partners["OpDiv"]

last_24h_multiple_partners

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
28,103.120.116.162,2026-04-22 00:00:00+00:00,Address,83,3,0.995452,0,0,"CMS, FDA, OS, VA",Incident:INC9385644,NaN,180,590,644,high,[2026-04-20] Severity: high. VT score: 11. Con...,<NA>,"CMS, FDA, OS, VA"
45,103.140.62.43,2026-04-22 00:00:00+00:00,Address,24,5,0.998685,0,0,"CMS, VA",NaN,NaN,180,590,149,low,[2026-04-20] Severity: low. VT score: 3. Conte...,Group Id: 315446,"CMS, VA"
213,104.41.137.249,2026-04-22 00:00:00+00:00,Address,11,3,0.999397,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9486336;Incident:INC9486332,NaN,480,740,217,medium,[2026-04-20] Severity: low. VT score: 1. Conte...,<NA>,"CMS, FDA, NIH, OS, VA"
214,104.41.143.27,2026-04-22 00:00:00+00:00,Address,55,3,0.996986,0,0,"CMS, FDA, OS, VA",Incident:INC9421495,NaN,520,760,228,medium,[2026-04-20] Severity: low. VT score: 2. Conte...,<NA>,"CMS, FDA, OS, VA"
265,111.175.88.6,2026-04-22 00:00:00+00:00,Address,6,3,0.999671,0,0,"CMS, IHS, OS",Incident:INC9460392,NaN,180,527,667,high,[2026-04-20] Severity: high. VT score: 12. Con...,<NA>,"CMS, IHS, OS"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3299,91.196.152.43,2026-04-22 00:00:00+00:00,Address,96,3,0.994740,0,0,"CMS, FDA, OS",NaN,NaN,260,509,619,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, OS"
3300,91.196.152.88,2026-04-22 00:00:00+00:00,Address,102,3,0.994411,0,0,"CMS, FDA, NIH, OS",NaN,NaN,180,590,425,medium,[2026-04-20] Severity: medium. VT score: 9. Co...,<NA>,"CMS, FDA, NIH, OS"
3307,91.224.92.147,2026-04-22 00:00:00+00:00,Address,27,3,0.998521,0,0,"CDC, CMS, FDA, IHS, NIH, OS, VA",Incident:INC9480324,NaN,180,590,639,high,[2026-04-20] Severity: high. VT score: 12. Con...,<NA>,"CDC, CMS, FDA, IHS, NIH, OS, VA"
3334,93.100.91.16,2026-04-22 00:00:00+00:00,Address,160,3,0.991233,0,0,"CMS, FDA, OS",Incident:INC9282086,NaN,590,674,468,medium,[2026-04-20] Severity: medium. VT score: 6. Co...,<NA>,"CMS, FDA, OS"


In [54]:
# Filter multi-partner, last-24h indicators to VT score >= 10 based on Explanation text
vt_scores = last_24h_multiple_partners['Explanation'].str.extract(r'VT score:\s*(\d+)', expand=False)
vt_scores = pd.to_numeric(vt_scores, errors='coerce')

last_24h_multi_partners_vt15 = last_24h_multiple_partners[vt_scores >= 14]

last_24h_multi_partners_vt15

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
279,112.31.180.128,2026-04-22 00:00:00+00:00,Address,50,3,0.997260,0,0,"CMS, FDA, OS",Incident:6755399484001244,NaN,180,368,613,high,[2026-04-20] Severity: high. VT score: 15. Con...,"Group Id: 6755399484001244, Group Id: 56294995...","CMS, FDA, OS"
949,172.94.9.253,2026-04-22 00:00:00+00:00,Address,43,3,0.997644,0,0,"CMS, FDA, IHS, NIH, OS",NaN,NaN,180,469,689,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, IHS, NIH, OS"
992,176.65.148.51,2026-04-22 00:00:00+00:00,Address,40,3,0.997808,0,0,"CMS, FDA, IHS, OS, VA",NaN,NaN,180,416,598,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, IHS, OS, VA"
1196,185.226.197.47,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,707,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, NIH, OS, VA"
1198,185.226.197.50,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,688,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, NIH, OS, VA"
1396,193.142.146.230,2026-04-22 00:00:00+00:00,Address,143,3,0.992164,0,0,"CMS, FDA, IHS, NIH, OS, VA",NaN,NaN,180,447,692,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, IHS, NIH, OS, VA"
1473,193.32.162.145,2026-04-22 00:00:00+00:00,Address,188,3,0.989699,0,0,"CMS, FDA, HHS, NIH, OS",Incident:INC9263763,NaN,820,736,792,high,[2026-04-20] Severity: critical. VT score: 14....,Group Id: 5629499579002100,"CMS, FDA, HHS, NIH, OS"
1520,195.178.110.188,2026-04-22 00:00:00+00:00,Address,14,3,0.999233,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,679,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"CMS, IHS, NIH, OS"
1540,196.11.86.134,2026-04-22 00:00:00+00:00,Address,26,3,0.998575,0,0,"CMS, FDA, OS",NaN,NaN,180,433,511,high,[2026-04-20] Severity: high. VT score: 15. Con...,Group Id: 5629499582001531,"CMS, FDA, OS"
1973,207.90.244.13,2026-04-22 00:00:00+00:00,Address,125,3,0.993151,0,0,"FDA, OS, VA",Incident:INC9337221,NaN,630,593,792,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"FDA, OS, VA"


In [55]:
# Keep only high or critical indicators from the VT>=10, multi-partner, last-24h set
final_indicators = last_24h_multi_partners_vt15[last_24h_multi_partners_vt15['Severity'].isin(['high', 'critical'])]

final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,OpDiv
279,112.31.180.128,2026-04-22 00:00:00+00:00,Address,50,3,0.997260,0,0,"CMS, FDA, OS",Incident:6755399484001244,NaN,180,368,613,high,[2026-04-20] Severity: high. VT score: 15. Con...,"Group Id: 6755399484001244, Group Id: 56294995...","CMS, FDA, OS"
949,172.94.9.253,2026-04-22 00:00:00+00:00,Address,43,3,0.997644,0,0,"CMS, FDA, IHS, NIH, OS",NaN,NaN,180,469,689,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, IHS, NIH, OS"
992,176.65.148.51,2026-04-22 00:00:00+00:00,Address,40,3,0.997808,0,0,"CMS, FDA, IHS, OS, VA",NaN,NaN,180,416,598,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, IHS, OS, VA"
1196,185.226.197.47,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,707,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, NIH, OS, VA"
1198,185.226.197.50,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,688,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"CMS, FDA, NIH, OS, VA"
1396,193.142.146.230,2026-04-22 00:00:00+00:00,Address,143,3,0.992164,0,0,"CMS, FDA, IHS, NIH, OS, VA",NaN,NaN,180,447,692,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,"CMS, FDA, IHS, NIH, OS, VA"
1473,193.32.162.145,2026-04-22 00:00:00+00:00,Address,188,3,0.989699,0,0,"CMS, FDA, HHS, NIH, OS",Incident:INC9263763,NaN,820,736,792,high,[2026-04-20] Severity: critical. VT score: 14....,Group Id: 5629499579002100,"CMS, FDA, HHS, NIH, OS"
1520,195.178.110.188,2026-04-22 00:00:00+00:00,Address,14,3,0.999233,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,679,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"CMS, IHS, NIH, OS"
1540,196.11.86.134,2026-04-22 00:00:00+00:00,Address,26,3,0.998575,0,0,"CMS, FDA, OS",NaN,NaN,180,433,511,high,[2026-04-20] Severity: high. VT score: 15. Con...,Group Id: 5629499582001531,"CMS, FDA, OS"
1973,207.90.244.13,2026-04-22 00:00:00+00:00,Address,125,3,0.993151,0,0,"FDA, OS, VA",Incident:INC9337221,NaN,630,593,792,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,"FDA, OS, VA"


In [56]:
import pandas as pd

# Load external tags data
tags_path = r"Z:\HTOC\Data_Analytics\Data\Observed_Tags\htoc_observed_indicator_tags.csv"
tags_df = pd.read_csv(tags_path)

# The indicator column in the tags CSV could be e.g. 'Indicator' or 'indicator'
tags_indicator_col = None
for col in tags_df.columns:
    if str(col).lower() == 'indicator':
        tags_indicator_col = col
        break
if tags_indicator_col is None:
    raise ValueError("Could not find an 'Indicator' column in the tags CSV.")

# The tags field might be called 'Tags', 'tags', or similar
# The tags field might be called 'Tags', 'tags', 'Tag', 'tag', etc.
tags_value_col = None
for col in tags_df.columns:
    if str(col).lower() in ('tags', 'tag'):
        tags_value_col = col
        break
if tags_value_col is None:
    raise ValueError(
        f"Could not find a 'Tag' or 'Tags' column in the tags CSV. "
        f"Available columns: {list(tags_df.columns)}"
    )
# For fast lookup, set up a mapping of indicator -> tags value.
indicator_to_tags = tags_df.set_index(tags_indicator_col)[tags_value_col].to_dict()

# Prepare 'Tags' values for final_indicators
final_tags = final_indicators['Indicator'].map(indicator_to_tags)

# Insert the 'Tags' column as the second to last column
final_cols = list(final_indicators.columns)
if 'Tags' in final_cols:
    final_cols.remove('Tags')
second_to_last_idx = -1 if len(final_cols) == 0 else -1
new_cols = final_cols[:second_to_last_idx] + ['Tags'] + final_cols[second_to_last_idx:]

final_indicators['Tags'] = final_tags
final_indicators = final_indicators[new_cols]
final_indicators


,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv
279,112.31.180.128,2026-04-22 00:00:00+00:00,Address,50,3,0.997260,0,0,"CMS, FDA, OS",Incident:6755399484001244,NaN,180,368,613,high,[2026-04-20] Severity: high. VT score: 15. Con...,"Group Id: 6755399484001244, Group Id: 56294995...",NaN,"CMS, FDA, OS"
949,172.94.9.253,2026-04-22 00:00:00+00:00,Address,43,3,0.997644,0,0,"CMS, FDA, IHS, NIH, OS",NaN,NaN,180,469,689,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,SOAR Indicator PB,"CMS, FDA, IHS, NIH, OS"
992,176.65.148.51,2026-04-22 00:00:00+00:00,Address,40,3,0.997808,0,0,"CMS, FDA, IHS, OS, VA",NaN,NaN,180,416,598,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,SOAR Indicator PB,"CMS, FDA, IHS, OS, VA"
1196,185.226.197.47,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,707,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,Web Scanner,"CMS, FDA, NIH, OS, VA"
1198,185.226.197.50,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,688,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, NIH, OS, VA"
1396,193.142.146.230,2026-04-22 00:00:00+00:00,Address,143,3,0.992164,0,0,"CMS, FDA, IHS, NIH, OS, VA",NaN,NaN,180,447,692,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,SOAR Indicator PB,"CMS, FDA, IHS, NIH, OS, VA"
1473,193.32.162.145,2026-04-22 00:00:00+00:00,Address,188,3,0.989699,0,0,"CMS, FDA, HHS, NIH, OS",Incident:INC9263763,NaN,820,736,792,high,[2026-04-20] Severity: critical. VT score: 14....,Group Id: 5629499579002100,SOAR Indicator PB,"CMS, FDA, HHS, NIH, OS"
1520,195.178.110.188,2026-04-22 00:00:00+00:00,Address,14,3,0.999233,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,679,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, IHS, NIH, OS"
1540,196.11.86.134,2026-04-22 00:00:00+00:00,Address,26,3,0.998575,0,0,"CMS, FDA, OS",NaN,NaN,180,433,511,high,[2026-04-20] Severity: high. VT score: 15. Con...,Group Id: 5629499582001531,NaN,"CMS, FDA, OS"
1973,207.90.244.13,2026-04-22 00:00:00+00:00,Address,125,3,0.993151,0,0,"FDA, OS, VA",Incident:INC9337221,NaN,630,593,792,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,CDN,"FDA, OS, VA"


In [57]:
import pandas as pd

# Helper to see if an indicator has an I&W tag
def has_iw(tags_value):
    """
    tags_value is typically a list of dicts from ThreatConnect, e.g.:
    [{'name': 'I&W'}, {'name': 'something else'}, ...]
    """
    if tags_value is None or (isinstance(tags_value, float) and pd.isna(tags_value)):
        return False

    if not isinstance(tags_value, (list, tuple)):
        return False

    for t in tags_value:
        try:
            if isinstance(t, dict):
                name = str(t.get('name', '')).strip()
            else:
                name = str(t).strip()

            if name.lower() in {"i&w", "i & w", "iw"}:
                return True
        except Exception:
            continue
    return False

# 1) Add has_iw flag to observed_src if tags.data exists
if 'tags.data' in observed_src.columns:
    observed_src['has_iw'] = observed_src['tags.data'].apply(has_iw)
else:
    observed_src['has_iw'] = False

# 2) Collapse to one flag per indicator
iw_per_indicator = (
    observed_src.groupby('indicator', dropna=False)['has_iw']
    .max()  # any True -> True
    .reset_index()
    .rename(columns={'indicator': 'Indicator', 'has_iw': 'Reported I&W?_raw'})
)

# 3) Drop ANY existing Reported I&W? variants (_x, _y, etc.)
cols_to_drop = [c for c in final_indicators.columns if c.startswith('Reported I&W?')]
final_indicators = final_indicators.drop(columns=cols_to_drop, errors='ignore')

# 4) Merge once, with a temporary raw boolean column
final_indicators = final_indicators.merge(
    iw_per_indicator,
    on='Indicator',
    how='left'
)

# 5) Convert to Yes/No, defaulting missing to 'No'
final_indicators['Reported I&W?'] = (
    final_indicators['Reported I&W?_raw']
    .fillna(False)
    .map({True: 'Yes', False: 'No'})
)

# 6) Drop the temporary raw column
final_indicators = final_indicators.drop(columns=['Reported I&W?_raw'])

# Rename column 'HTOC Threat Score' to 'PRISM Score' if it exists
if "HTOC Threat Score" in final_indicators.columns:
    final_indicators = final_indicators.rename(columns={"HTOC Threat Score": "PRISM Score"})


final_indicators

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation,Associated Groups,Tags,OpDiv,Reported I&W?
0,112.31.180.128,2026-04-22 00:00:00+00:00,Address,50,3,0.997260,0,0,"CMS, FDA, OS",Incident:6755399484001244,NaN,180,368,613,high,[2026-04-20] Severity: high. VT score: 15. Con...,"Group Id: 6755399484001244, Group Id: 56294995...",NaN,"CMS, FDA, OS",Yes
1,172.94.9.253,2026-04-22 00:00:00+00:00,Address,43,3,0.997644,0,0,"CMS, FDA, IHS, NIH, OS",NaN,NaN,180,469,689,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,SOAR Indicator PB,"CMS, FDA, IHS, NIH, OS",No
2,176.65.148.51,2026-04-22 00:00:00+00:00,Address,40,3,0.997808,0,0,"CMS, FDA, IHS, OS, VA",NaN,NaN,180,416,598,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,SOAR Indicator PB,"CMS, FDA, IHS, OS, VA",No
3,185.226.197.47,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,707,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,Web Scanner,"CMS, FDA, NIH, OS, VA",No
4,185.226.197.50,2026-04-22 00:00:00+00:00,Address,97,3,0.994685,0,0,"CMS, FDA, NIH, OS, VA",Incident:INC9366814,NaN,680,777,688,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, FDA, NIH, OS, VA",No
5,193.142.146.230,2026-04-22 00:00:00+00:00,Address,143,3,0.992164,0,0,"CMS, FDA, IHS, NIH, OS, VA",NaN,NaN,180,447,692,high,[2026-04-20] Severity: high. VT score: 15. Con...,<NA>,SOAR Indicator PB,"CMS, FDA, IHS, NIH, OS, VA",No
6,193.32.162.145,2026-04-22 00:00:00+00:00,Address,188,3,0.989699,0,0,"CMS, FDA, HHS, NIH, OS",Incident:INC9263763,NaN,820,736,792,high,[2026-04-20] Severity: critical. VT score: 14....,Group Id: 5629499579002100,SOAR Indicator PB,"CMS, FDA, HHS, NIH, OS",Yes
7,195.178.110.188,2026-04-22 00:00:00+00:00,Address,14,3,0.999233,0,0,"CMS, IHS, NIH, OS",Incident:INC9480324,NaN,180,590,679,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,Web Scanner,"CMS, IHS, NIH, OS",No
8,196.11.86.134,2026-04-22 00:00:00+00:00,Address,26,3,0.998575,0,0,"CMS, FDA, OS",NaN,NaN,180,433,511,high,[2026-04-20] Severity: high. VT score: 15. Con...,Group Id: 5629499582001531,NaN,"CMS, FDA, OS",Yes
9,207.90.244.13,2026-04-22 00:00:00+00:00,Address,125,3,0.993151,0,0,"FDA, OS, VA",Incident:INC9337221,NaN,630,593,792,high,[2026-04-20] Severity: high. VT score: 14. Con...,<NA>,CDN,"FDA, OS, VA",No


In [59]:
from datetime import datetime

# Build dated output path
today_str = datetime.today().strftime('%Y%m%d')  # e.g. 20260316
output_path = rf"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\ThreatAssessI_W\ThreatAssessI_W_{today_str}.xlsx"

# Excel can't write timezone-aware datetimes; strip tz info before export
_dt_tz_cols = final_indicators.select_dtypes(include=["datetimetz"]).columns
for _c in _dt_tz_cols:
    final_indicators[_c] = final_indicators[_c].dt.tz_convert(None)

iw_col = "Reported I&W?"
if iw_col not in final_indicators.columns:
    raise KeyError(f"Missing required column '{iw_col}' for sheet split.")

final_iw_no = final_indicators[final_indicators[iw_col] == "No"].copy()
final_iw_yes = final_indicators[final_indicators[iw_col] == "Yes"].copy()

# Write to one workbook with two named sheets
with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
    final_iw_no.to_excel(writer, index=False, sheet_name="I&W_No")
    final_iw_yes.to_excel(writer, index=False, sheet_name="I&W_Yes")

    workbook = writer.book
    wrap_fmt = workbook.add_format({"text_wrap": True, "valign": "top"})

    # Set defaults first, then tune long text columns for each sheet
    for sheet_name, sheet_df in [("I&W_No", final_iw_no), ("I&W_Yes", final_iw_yes)]:
        worksheet = writer.sheets[sheet_name]
        worksheet.set_column(0, len(final_indicators.columns) - 1, 18)

        if "Explanation" in final_indicators.columns:
            _exp_idx = final_indicators.columns.get_loc("Explanation")
            worksheet.set_column(_exp_idx, _exp_idx, 100, wrap_fmt)

        if "Associated Groups" in final_indicators.columns:
            _ag_idx = final_indicators.columns.get_loc("Associated Groups")
            worksheet.set_column(_ag_idx, _ag_idx, 45, wrap_fmt)

output_path

'Z:\\HTOC\\Data_Analytics\\Data\\Threat Assessment Scores\\ThreatAssessI_W\\ThreatAssessI_W_20260422.xlsx'